In [1]:
print("Kernel is working")

Kernel is working


In [5]:
from pathlib import Path

project_path = Path("..")

print(project_path.resolve())

C:\Users\HARSHITHA\Desktop\foresight


In [6]:
for file in project_path.rglob("*.csv"):
    print(file)

..\data\processed\calendar_clean.csv
..\data\processed\inventory_snapshots_clean.csv
..\data\processed\sales_daily_clean.csv
..\data\processed\sku_master_clean.csv
..\data\raw\calendar.csv
..\data\raw\inventory_snapshots.csv
..\data\raw\sales_daily.csv
..\data\raw\sku_master.csv


In [7]:
import pandas as pd

sales = pd.read_csv("../data/processed/sales_daily_clean.csv")

sales["Date"] = pd.to_datetime(sales["Date"])

print("Sales shape:", sales.shape)
print(sales.head())
print(sales.columns.tolist())

Sales shape: (36550, 6)
        Date     SKU  Units_Sold    Revenue    Price  Promotion
0 2024-01-01  SKU001           5   18320.25  3664.05          0
1 2024-01-01  SKU002          15   57085.35  3805.69          0
2 2024-01-01  SKU003           5   40391.15  8078.23          0
3 2024-01-01  SKU004           5   34307.85  6861.57          0
4 2024-01-01  SKU005          12  113918.64  9493.22          0
['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion']


In [8]:
sku = pd.read_csv("../data/processed/sku_master_clean.csv")

print("SKU shape:", sku.shape)
print(sku.head())
print(sku.columns.tolist())

SKU shape: (50, 8)
      SKU Product_Name    Category Subcategory Launch_Date  Cost_Price  \
0  SKU001  Product 001   Furniture       Chair  2022-04-09     1758.45   
1  SKU002  Product 002  Home Decor       Table  2024-05-01     3867.09   
2  SKU003  Product 003     Kitchen     Cushion  2023-12-22      589.48   
3  SKU004  Product 004    Lighting    Cookware  2023-04-28     1445.74   
4  SKU005  Product 005     Storage        Lamp  2023-04-22     5543.63   

   Selling_Price  Gross_Margin_Per_Unit  
0        3664.05                1905.60  
1        3805.69                 -61.40  
2        8078.23                7488.75  
3        6861.57                5415.83  
4        9493.22                3949.59  
['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit']


In [9]:
print("Sales columns:", sales.columns.tolist())
print("SKU columns:", sku.columns.tolist())

Sales columns: ['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion']
SKU columns: ['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit']


In [10]:
import pandas as pd
import numpy as np

sales = pd.read_csv("../data/processed/sales_daily_clean.csv")
sku = pd.read_csv("../data/processed/sku_master_clean.csv")

sales["Date"] = pd.to_datetime(sales["Date"])

print("Sales shape:", sales.shape)
print("SKU shape:", sku.shape)

print("\nSales columns:")
print(sales.columns.tolist())

print("\nSKU columns:")
print(sku.columns.tolist())

Sales shape: (36550, 6)
SKU shape: (50, 8)

Sales columns:
['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion']

SKU columns:
['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit']


In [11]:
weekly_sales = (
    sales.set_index("Date")
    .groupby("SKU")["Units_Sold"]
    .resample("W-MON")
    .sum()
    .reset_index()
)

weekly_sales = weekly_sales.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

print("Weekly dataset shape:", weekly_sales.shape)
print("Number of SKUs:", weekly_sales["SKU"].nunique())

print(
    "Date range:",
    weekly_sales["Date"].min(),
    "to",
    weekly_sales["Date"].max()
)

print(weekly_sales.head())

Weekly dataset shape: (5300, 3)
Number of SKUs: 50
Date range: 2024-01-01 00:00:00 to 2026-01-05 00:00:00
      SKU       Date  Units_Sold
0  SKU001 2024-01-01           5
1  SKU001 2024-01-08         113
2  SKU001 2024-01-15          88
3  SKU001 2024-01-22         115
4  SKU001 2024-01-29          90


In [12]:
weekly_sales["Baseline_Forecast"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(52)
)

print(
    weekly_sales[
        ["SKU", "Date", "Units_Sold", "Baseline_Forecast"]
    ].head(60)
)

       SKU       Date  Units_Sold  Baseline_Forecast
0   SKU001 2024-01-01           5                NaN
1   SKU001 2024-01-08         113                NaN
2   SKU001 2024-01-15          88                NaN
3   SKU001 2024-01-22         115                NaN
4   SKU001 2024-01-29          90                NaN
5   SKU001 2024-02-05         120                NaN
6   SKU001 2024-02-12         111                NaN
7   SKU001 2024-02-19         105                NaN
8   SKU001 2024-02-26         113                NaN
9   SKU001 2024-03-04         107                NaN
10  SKU001 2024-03-11         118                NaN
11  SKU001 2024-03-18         133                NaN
12  SKU001 2024-03-25         115                NaN
13  SKU001 2024-04-01         108                NaN
14  SKU001 2024-04-08         124                NaN
15  SKU001 2024-04-15         111                NaN
16  SKU001 2024-04-22         103                NaN
17  SKU001 2024-04-29         117             

In [13]:
baseline_test = weekly_sales.dropna(
    subset=["Baseline_Forecast"]
).copy()

baseline_wape = (
    (
        baseline_test["Units_Sold"]
        - baseline_test["Baseline_Forecast"]
    )
    .abs()
    .sum()
    /
    baseline_test["Units_Sold"].abs().sum()
)

print(
    "Seasonal-Naive WAPE:",
    round(baseline_wape * 100, 2),
    "%"
)

Seasonal-Naive WAPE: 13.29 %


In [14]:
weekly_sales = weekly_sales.sort_values(
    ["SKU", "Date"]
).copy()

weekly_sales["Year"] = weekly_sales["Date"].dt.year

weekly_sales["Month"] = weekly_sales["Date"].dt.month

weekly_sales["WeekOfYear"] = (
    weekly_sales["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

weekly_sales["Lag_1"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(1)
)

weekly_sales["Lag_4"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(4)
)

weekly_sales["Lag_13"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(13)
)

weekly_sales["Lag_52"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(52)
)

weekly_sales["Rolling_Mean_4"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .transform(
        lambda x: x.shift(1).rolling(4).mean()
    )
)

weekly_sales["Rolling_Mean_13"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .transform(
        lambda x: x.shift(1).rolling(13).mean()
    )
)

print(weekly_sales.head(20))

       SKU       Date  Units_Sold  Baseline_Forecast  Year  Month  WeekOfYear  \
0   SKU001 2024-01-01           5                NaN  2024      1           1   
1   SKU001 2024-01-08         113                NaN  2024      1           2   
2   SKU001 2024-01-15          88                NaN  2024      1           3   
3   SKU001 2024-01-22         115                NaN  2024      1           4   
4   SKU001 2024-01-29          90                NaN  2024      1           5   
5   SKU001 2024-02-05         120                NaN  2024      2           6   
6   SKU001 2024-02-12         111                NaN  2024      2           7   
7   SKU001 2024-02-19         105                NaN  2024      2           8   
8   SKU001 2024-02-26         113                NaN  2024      2           9   
9   SKU001 2024-03-04         107                NaN  2024      3          10   
10  SKU001 2024-03-11         118                NaN  2024      3          11   
11  SKU001 2024-03-18       

In [15]:
model_data = weekly_sales.dropna(
    subset=[
        "Lag_1",
        "Lag_4",
        "Lag_13",
        "Lag_52",
        "Rolling_Mean_4",
        "Rolling_Mean_13"
    ]
).copy()

print("Model dataset shape:", model_data.shape)
print("Number of SKUs:", model_data["SKU"].nunique())

Model dataset shape: (2700, 13)
Number of SKUs: 50


In [16]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.1


In [17]:
split_date = model_data["Date"].quantile(0.80)

train = model_data[
    model_data["Date"] <= split_date
].copy()

test = model_data[
    model_data["Date"] > split_date
].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train end:", train["Date"].max())
print("Test start:", test["Date"].min())

Train rows: 2200
Test rows: 500
Train end: 2025-10-27 00:00:00
Test start: 2025-11-03 00:00:00


In [18]:
features = [
    "Lag_1",
    "Lag_4",
    "Lag_13",
    "Lag_52",
    "Rolling_Mean_4",
    "Rolling_Mean_13",
    "Year",
    "Month",
    "WeekOfYear"
]

X_train = train[features]
y_train = train["Units_Sold"]

X_test = test[features]
y_test = test["Units_Sold"]

print("Features:")
print(features)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Features:
['Lag_1', 'Lag_4', 'Lag_13', 'Lag_52', 'Rolling_Mean_4', 'Rolling_Mean_13', 'Year', 'Month', 'WeekOfYear']

Training shape: (2200, 9)
Testing shape: (500, 9)


In [19]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

test["Model_Forecast"] = model.predict(
    X_test
)

test["Model_Forecast"] = (
    test["Model_Forecast"]
    .clip(lower=0)
)

print(
    test[
        [
            "SKU",
            "Date",
            "Units_Sold",
            "Model_Forecast"
        ]
    ].head(20)
)

        SKU       Date  Units_Sold  Model_Forecast
96   SKU001 2025-11-03          96           83.88
97   SKU001 2025-11-10          89           84.69
98   SKU001 2025-11-17          83           80.58
99   SKU001 2025-11-24          92           81.84
100  SKU001 2025-12-01          81           78.78
101  SKU001 2025-12-08          89           86.10
102  SKU001 2025-12-15          93           88.03
103  SKU001 2025-12-22          82           77.15
104  SKU001 2025-12-29          81           98.12
105  SKU001 2026-01-05          25           89.83
202  SKU002 2025-11-03          76           56.46
203  SKU002 2025-11-10          71           52.76
204  SKU002 2025-11-17          57           58.44
205  SKU002 2025-11-24          66           56.40
206  SKU002 2025-12-01          66           61.66
207  SKU002 2025-12-08          50           62.15
208  SKU002 2025-12-15          55           56.84
209  SKU002 2025-12-22          51           50.80
210  SKU002 2025-12-29         

In [20]:
model_wape = (
    (
        test["Units_Sold"]
        - test["Model_Forecast"]
    )
    .abs()
    .sum()
    /
    test["Units_Sold"].abs().sum()
)

print(
    "Random Forest WAPE:",
    round(model_wape * 100, 2),
    "%"
)

Random Forest WAPE: 19.92 %


In [21]:
comparison = weekly_sales.copy()

comparison["Baseline_Forecast"] = (
    comparison
    .groupby("SKU")["Units_Sold"]
    .shift(52)
)

comparison = comparison[
    comparison["Date"] > split_date
].dropna(
    subset=["Baseline_Forecast"]
)

baseline_wape_test = (
    (
        comparison["Units_Sold"]
        - comparison["Baseline_Forecast"]
    )
    .abs()
    .sum()
    /
    comparison["Units_Sold"].abs().sum()
)

print(
    "Seasonal-Naive Test WAPE:",
    round(baseline_wape_test * 100, 2),
    "%"
)

print(
    "Random Forest WAPE:",
    round(model_wape * 100, 2),
    "%"
)

Seasonal-Naive Test WAPE: 20.53 %
Random Forest WAPE: 19.92 %


In [22]:
improvement = (
    (baseline_wape_test - model_wape)
    / baseline_wape_test
) * 100

print(
    "WAPE improvement over Seasonal-Naive:",
    round(improvement, 2),
    "%"
)

WAPE improvement over Seasonal-Naive: 2.97 %


In [23]:
from sklearn.ensemble import RandomForestRegressor

origins = sorted(
    model_data["Date"].unique()
)

origins = origins[-4:]

results = []

for origin in origins:

    train_cv = model_data[
        model_data["Date"] < origin
    ].copy()

    test_cv = model_data[
        model_data["Date"] == origin
    ].copy()

    if len(train_cv) == 0 or len(test_cv) == 0:
        continue

    X_train_cv = train_cv[features]
    y_train_cv = train_cv["Units_Sold"]

    X_test_cv = test_cv[features]
    y_test_cv = test_cv["Units_Sold"]

    cv_model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    cv_model.fit(
        X_train_cv,
        y_train_cv
    )

    prediction = cv_model.predict(
        X_test_cv
    )

    prediction = np.clip(
        prediction,
        0,
        None
    )

    results.append({
        "Origin": origin,
        "Actual": y_test_cv.sum(),
        "Forecast": prediction.sum()
    })

rolling_results = pd.DataFrame(results)

rolling_results["Absolute_Error"] = (
    rolling_results["Actual"]
    - rolling_results["Forecast"]
).abs()

rolling_wape = (
    rolling_results["Absolute_Error"].sum()
    /
    rolling_results["Actual"].abs().sum()
)

print(rolling_results)

print(
    "Rolling-origin WAPE:",
    round(rolling_wape * 100, 2),
    "%"
)

      Origin  Actual  Forecast  Absolute_Error
0 2025-12-15    4364   4300.33           63.67
1 2025-12-22    4334   4309.97           24.03
2 2025-12-29    4311   4526.19          215.19
3 2026-01-05    1150   4649.37         3499.37
Rolling-origin WAPE: 26.85 %


In [24]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

           Feature  Importance
4   Rolling_Mean_4    0.833402
3           Lag_52    0.123779
5  Rolling_Mean_13    0.010516
0            Lag_1    0.009078
8       WeekOfYear    0.008629
2           Lag_13    0.006423
1            Lag_4    0.005484
7            Month    0.002671
6             Year    0.000018


In [25]:
last_date = weekly_sales["Date"].max()

print("Last historical date:", last_date)

Last historical date: 2026-01-05 00:00:00


In [26]:
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(weeks=1),
    periods=6,
    freq="W-MON"
)

print(future_dates)

DatetimeIndex(['2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02',
               '2026-02-09', '2026-02-16'],
              dtype='datetime64[us]', freq='W-MON')


In [27]:
history = weekly_sales[
    ["SKU", "Date", "Units_Sold"]
].copy()

future_results = []

skus = history["SKU"].unique()

for future_date in future_dates:

    for current_sku in skus:

        sku_history = history[
            history["SKU"] == current_sku
        ].sort_values("Date")

        if len(sku_history) < 52:
            continue

        values = sku_history["Units_Sold"].values

        lag_1 = values[-1]
        lag_4 = values[-4]
        lag_13 = values[-13]
        lag_52 = values[-52]

        rolling_4 = values[-4:].mean()
        rolling_13 = values[-13:].mean()

        row = pd.DataFrame([{
            "Lag_1": lag_1,
            "Lag_4": lag_4,
            "Lag_13": lag_13,
            "Lag_52": lag_52,
            "Rolling_Mean_4": rolling_4,
            "Rolling_Mean_13": rolling_13,
            "Year": future_date.year,
            "Month": future_date.month,
            "WeekOfYear": int(
                future_date.isocalendar().week
            )
        }])

        forecast = model.predict(
            row[features]
        )[0]

        forecast = max(0, forecast)

        future_results.append({
            "SKU": current_sku,
            "Date": future_date,
            "Forecast_Units": forecast
        })

        history = pd.concat([
            history,
            pd.DataFrame([{
                "SKU": current_sku,
                "Date": future_date,
                "Units_Sold": forecast
            }])
        ], ignore_index=True)

In [28]:
future_forecast = pd.DataFrame(
    future_results
)

print(
    "Future forecast shape:",
    future_forecast.shape
)

print(
    future_forecast.head(20)
)

Future forecast shape: (300, 3)
       SKU       Date  Forecast_Units
0   SKU001 2026-01-12           79.19
1   SKU002 2026-01-12           48.42
2   SKU003 2026-01-12           33.56
3   SKU004 2026-01-12           33.26
4   SKU005 2026-01-12           94.15
5   SKU006 2026-01-12           36.71
6   SKU007 2026-01-12          158.24
7   SKU008 2026-01-12           93.60
8   SKU009 2026-01-12           59.17
9   SKU010 2026-01-12           92.21
10  SKU011 2026-01-12           19.64
11  SKU012 2026-01-12          159.33
12  SKU013 2026-01-12           80.42
13  SKU014 2026-01-12          109.86
14  SKU015 2026-01-12           18.83
15  SKU016 2026-01-12           80.98
16  SKU017 2026-01-12           84.70
17  SKU018 2026-01-12          157.81
18  SKU019 2026-01-12           93.76
19  SKU020 2026-01-12           85.60


In [30]:
from pathlib import Path

Path("../reports").mkdir(
    exist_ok=True
)

In [31]:
future_forecast.to_csv(
    "../reports/future_6_week_forecast.csv",
    index=False
)

print("6-week forecast saved successfully.")

6-week forecast saved successfully.


In [32]:
evaluation = pd.DataFrame({
    "Model": [
        "Seasonal-Naive",
        "Random Forest",
        "Rolling-Origin Random Forest"
    ],
    "WAPE": [
        baseline_wape * 100,
        model_wape * 100,
        rolling_wape * 100
    ]
})

print(evaluation)

                          Model       WAPE
0                Seasonal-Naive  13.291404
1                 Random Forest  19.916400
2  Rolling-Origin Random Forest  26.854015


In [33]:
evaluation.to_csv(
    "../reports/forecast_model_evaluation.csv",
    index=False
)

print("Model evaluation saved.")

Model evaluation saved.


In [34]:
print(
    sales[sales["Date"] >= "2025-12-29"]
    .groupby(sales["Date"].dt.to_period("W-MON"))["Units_Sold"]
    .sum()
)

print(
    sales["Date"].max()
)

Date
2025-12-23/2025-12-29     578
2025-12-30/2026-01-05    1150
Freq: W-MON, Name: Units_Sold, dtype: int64
2025-12-31 00:00:00


In [35]:
complete_weekly_sales = weekly_sales[
    weekly_sales["Date"] < pd.Timestamp("2026-01-05")
].copy()

print("Complete weekly dataset:")
print("Shape:", complete_weekly_sales.shape)
print("Last date:", complete_weekly_sales["Date"].max())

Complete weekly dataset:
Shape: (5250, 13)
Last date: 2025-12-29 00:00:00


In [36]:
complete_weekly_sales["Baseline_Forecast"] = (
    complete_weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(52)
)

baseline_complete = complete_weekly_sales.dropna(
    subset=["Baseline_Forecast"]
)

baseline_wape_complete = (
    (
        baseline_complete["Units_Sold"]
        - baseline_complete["Baseline_Forecast"]
    )
    .abs()
    .sum()
    /
    baseline_complete["Units_Sold"].abs().sum()
)

print(
    "Seasonal-Naive WAPE:",
    round(baseline_wape_complete * 100, 2),
    "%"
)

Seasonal-Naive WAPE: 11.97 %


In [37]:
print(
    "Last complete week:",
    complete_weekly_sales["Date"].max()
)

Last complete week: 2025-12-29 00:00:00


In [38]:
complete_model_data = complete_weekly_sales.dropna(
    subset=[
        "Lag_1",
        "Lag_4",
        "Lag_13",
        "Lag_52",
        "Rolling_Mean_4",
        "Rolling_Mean_13"
    ]
).copy()

print("Complete model dataset shape:", complete_model_data.shape)

Complete model dataset shape: (2650, 13)


In [39]:
split_date_complete = complete_model_data["Date"].quantile(0.80)

train_complete = complete_model_data[
    complete_model_data["Date"] <= split_date_complete
].copy()

test_complete = complete_model_data[
    complete_model_data["Date"] > split_date_complete
].copy()

print("Train rows:", len(train_complete))
print("Test rows:", len(test_complete))

print("Train end:", train_complete["Date"].max())
print("Test start:", test_complete["Date"].min())

Train rows: 2150
Test rows: 500
Train end: 2025-10-20 00:00:00
Test start: 2025-10-27 00:00:00


In [40]:
features = [
    "Lag_1",
    "Lag_4",
    "Lag_13",
    "Lag_52",
    "Rolling_Mean_4",
    "Rolling_Mean_13",
    "Year",
    "Month",
    "WeekOfYear"
]

X_train_complete = train_complete[features]
y_train_complete = train_complete["Units_Sold"]

X_test_complete = test_complete[features]
y_test_complete = test_complete["Units_Sold"]

corrected_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

corrected_model.fit(
    X_train_complete,
    y_train_complete
)

test_complete["Model_Forecast"] = (
    corrected_model.predict(X_test_complete)
)

test_complete["Model_Forecast"] = (
    test_complete["Model_Forecast"].clip(lower=0)
)

print(
    test_complete[
        ["SKU", "Date", "Units_Sold", "Model_Forecast"]
    ].head(10)
)

        SKU       Date  Units_Sold  Model_Forecast
95   SKU001 2025-10-27          77           87.68
96   SKU001 2025-11-03          96           82.66
97   SKU001 2025-11-10          89           82.36
98   SKU001 2025-11-17          83           77.24
99   SKU001 2025-11-24          92           82.20
100  SKU001 2025-12-01          81           76.77
101  SKU001 2025-12-08          89           83.23
102  SKU001 2025-12-15          93           84.66
103  SKU001 2025-12-22          82           75.20
104  SKU001 2025-12-29          81           94.82


In [41]:
corrected_model_wape = (
    (
        test_complete["Units_Sold"]
        - test_complete["Model_Forecast"]
    )
    .abs()
    .sum()
    /
    test_complete["Units_Sold"].abs().sum()
)

print(
    "Corrected Random Forest WAPE:",
    round(corrected_model_wape * 100, 2),
    "%"
)

Corrected Random Forest WAPE: 10.99 %


In [42]:
comparison_complete = pd.DataFrame({
    "Model": [
        "Seasonal-Naive",
        "Random Forest"
    ],
    "WAPE": [
        baseline_wape_complete * 100,
        corrected_model_wape * 100
    ]
})

print(comparison_complete)

            Model       WAPE
0  Seasonal-Naive  11.974638
1   Random Forest  10.989114


In [43]:
improvement_complete = (
    (
        baseline_wape_complete
        - corrected_model_wape
    )
    /
    baseline_wape_complete
) * 100

print(
    "WAPE improvement:",
    round(improvement_complete, 2),
    "%"
)

WAPE improvement: 8.23 %


In [44]:
origins = sorted(complete_model_data["Date"].unique())[-4:]

rolling_results = []

for origin in origins:

    train_cv = complete_model_data[
        complete_model_data["Date"] < origin
    ].copy()

    test_cv = complete_model_data[
        complete_model_data["Date"] == origin
    ].copy()

    X_train_cv = train_cv[features]
    y_train_cv = train_cv["Units_Sold"]

    X_test_cv = test_cv[features]
    y_test_cv = test_cv["Units_Sold"]

    cv_model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    cv_model.fit(X_train_cv, y_train_cv)

    prediction = cv_model.predict(X_test_cv)
    prediction = np.clip(prediction, 0, None)

    rolling_results.append({
        "Origin": origin,
        "Actual": y_test_cv.sum(),
        "Forecast": prediction.sum()
    })

rolling_results_complete = pd.DataFrame(rolling_results)

rolling_results_complete["Absolute_Error"] = (
    rolling_results_complete["Actual"]
    - rolling_results_complete["Forecast"]
).abs()

rolling_wape_complete = (
    rolling_results_complete["Absolute_Error"].sum()
    /
    rolling_results_complete["Actual"].abs().sum()
)

print(rolling_results_complete)

print(
    "Corrected Rolling-Origin WAPE:",
    round(rolling_wape_complete * 100, 2),
    "%"
)

      Origin  Actual  Forecast  Absolute_Error
0 2025-12-08    4368   4298.88           69.12
1 2025-12-15    4364   4300.33           63.67
2 2025-12-22    4334   4309.97           24.03
3 2025-12-29    4311   4526.19          215.19
Corrected Rolling-Origin WAPE: 2.14 %


In [45]:
evaluation_final = pd.DataFrame({
    "Model": [
        "Seasonal-Naive",
        "Random Forest",
        "Rolling-Origin Random Forest"
    ],
    "WAPE": [
        baseline_wape_complete * 100,
        corrected_model_wape * 100,
        rolling_wape_complete * 100
    ]
})

evaluation_final.to_csv(
    "../reports/forecast_model_evaluation.csv",
    index=False
)

print(evaluation_final)

                          Model       WAPE
0                Seasonal-Naive  11.974638
1                 Random Forest  10.989114
2  Rolling-Origin Random Forest   2.140818


In [46]:
print("Future forecast shape:", future_forecast.shape)
print("Forecast dates:")
print(sorted(future_forecast["Date"].unique()))

print("\nForecast file:")
print("../reports/future_6_week_forecast.csv")

Future forecast shape: (300, 3)
Forecast dates:
[Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-19 00:00:00'), Timestamp('2026-01-26 00:00:00'), Timestamp('2026-02-02 00:00:00'), Timestamp('2026-02-09 00:00:00'), Timestamp('2026-02-16 00:00:00')]

Forecast file:
../reports/future_6_week_forecast.csv


In [47]:
inventory = pd.read_csv(
    "../data/processed/inventory_snapshots_clean.csv"
)

print("Inventory shape:", inventory.shape)
print("\nInventory columns:")
print(inventory.columns.tolist())

print("\nFirst 5 rows:")
print(inventory.head())

Inventory shape: (1200, 8)

Inventory columns:
['Snapshot_Date', 'SKU', 'Current_Stock', 'On_Order', 'Lead_Time_Days', 'Safety_Stock', 'Reorder_Point', 'Inventory_Value']

First 5 rows:
  Snapshot_Date     SKU  Current_Stock  On_Order  Lead_Time_Days  \
0    2024-01-01  SKU001             16        23              11   
1    2024-01-01  SKU002             29        12               4   
2    2024-01-01  SKU003             34        23              14   
3    2024-01-01  SKU004             34         4               7   
4    2024-01-01  SKU005             23         5               5   

   Safety_Stock  Reorder_Point  Inventory_Value  
0             7             18         58420.96  
1             5             10        163983.40  
2             6             20        209060.22  
3             5             12         42257.92  
4             6             11        170945.89  


In [48]:
print("SKU master columns:")
print(sku.columns.tolist())

SKU master columns:
['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit']


In [49]:
inventory["Snapshot_Date"] = pd.to_datetime(inventory["Snapshot_Date"])

print("Inventory date range:")
print(inventory["Snapshot_Date"].min())
print(inventory["Snapshot_Date"].max())

print("\nNumber of SKUs:")
print(inventory["SKU"].nunique())

print("\nLatest snapshot:")
print(inventory["Snapshot_Date"].max())

Inventory date range:
2024-01-01 00:00:00
2025-12-01 00:00:00

Number of SKUs:
50

Latest snapshot:
2025-12-01 00:00:00


In [50]:
# D4 - Risk Scoring

# 1. Get latest inventory snapshot
latest_date = inventory["Snapshot_Date"].max()

latest_inventory = inventory[
    inventory["Snapshot_Date"] == latest_date
].copy()

# 2. Calculate total 6-week forecast demand for each SKU
forecast_summary = (
    future_forecast
    .groupby("SKU")["Forecast_Units"]
    .sum()
    .reset_index()
    .rename(columns={"Forecast_Units": "Forecast_6W_Units"})
)

# 3. Calculate demand during lead time
# Convert lead time from days to approximate weeks
latest_inventory["Lead_Time_Weeks"] = (
    latest_inventory["Lead_Time_Days"] / 7
)

latest_inventory["Lead_Time_Demand"] = (
    latest_inventory["Lead_Time_Weeks"]
    * latest_inventory["Reorder_Point"]
)

# 4. Merge forecast + inventory + SKU pricing
risk_df = latest_inventory.merge(
    forecast_summary,
    on="SKU",
    how="left"
)

risk_df = risk_df.merge(
    sku[
        [
            "SKU",
            "Product_Name",
            "Category",
            "Subcategory",
            "Cost_Price",
            "Selling_Price",
            "Gross_Margin_Per_Unit"
        ]
    ],
    on="SKU",
    how="left"
)

# 5. Projected stock after 6 weeks
risk_df["Projected_Stock_6W"] = (
    risk_df["Current_Stock"]
    + risk_df["On_Order"]
    - risk_df["Forecast_6W_Units"]
)

# 6. Stockout risk
risk_df["Stockout_Risk"] = np.where(
    risk_df["Current_Stock"] + risk_df["On_Order"]
    < risk_df["Forecast_6W_Units"],
    "High",
    "Low"
)

# 7. Overstock risk
# Flag when projected stock is more than 150% of forecast demand
risk_df["Overstock_Risk"] = np.where(
    risk_df["Projected_Stock_6W"]
    > (risk_df["Forecast_6W_Units"] * 1.5),
    "High",
    "Low"
)

# 8. Decisioning grid
def get_action(row):
    if row["Stockout_Risk"] == "High" and row["Overstock_Risk"] == "Low":
        return "Reorder now"
    elif row["Stockout_Risk"] == "Low" and row["Overstock_Risk"] == "High":
        return "Markdown / clear"
    elif row["Stockout_Risk"] == "High" and row["Overstock_Risk"] == "High":
        return "Watch / volatile"
    else:
        return "Healthy"

risk_df["Recommended_Action"] = risk_df.apply(
    get_action,
    axis=1
)

# 9. Rupee value at stake
risk_df["Sales_At_Risk_INR"] = np.where(
    risk_df["Stockout_Risk"] == "High",
    np.maximum(
        risk_df["Forecast_6W_Units"]
        - risk_df["Current_Stock"]
        - risk_df["On_Order"],
        0
    ) * risk_df["Selling_Price"],
    0
)

risk_df["Locked_Capital_INR"] = np.where(
    risk_df["Overstock_Risk"] == "High",
    np.maximum(
        risk_df["Projected_Stock_6W"]
        - risk_df["Forecast_6W_Units"],
        0
    ) * risk_df["Cost_Price"],
    0
)

# 10. Total value at stake
risk_df["Value_At_Stake_INR"] = (
    risk_df["Sales_At_Risk_INR"]
    + risk_df["Locked_Capital_INR"]
)

# 11. Display results
risk_columns = [
    "SKU",
    "Product_Name",
    "Category",
    "Current_Stock",
    "On_Order",
    "Forecast_6W_Units",
    "Projected_Stock_6W",
    "Stockout_Risk",
    "Overstock_Risk",
    "Recommended_Action",
    "Sales_At_Risk_INR",
    "Locked_Capital_INR",
    "Value_At_Stake_INR"
]

print(risk_df[risk_columns].head(10).to_string(index=False))

print("\nRisk summary:")
print(risk_df["Recommended_Action"].value_counts())

   SKU Product_Name   Category  Current_Stock  On_Order  Forecast_6W_Units  Projected_Stock_6W Stockout_Risk Overstock_Risk Recommended_Action  Sales_At_Risk_INR  Locked_Capital_INR  Value_At_Stake_INR
SKU001  Product 001  Furniture            771       147             492.35              425.65           Low            Low            Healthy             0.0000              0.0000              0.0000
SKU002  Product 002 Home Decor            526        29             316.89              238.11           Low            Low            Healthy             0.0000              0.0000              0.0000
SKU003  Product 003    Kitchen            349        12             210.18              150.82           Low            Low            Healthy             0.0000              0.0000              0.0000
SKU004  Product 004   Lighting            259       254             194.39              318.61           Low           High   Markdown / clear             0.0000         179589.8228         17

In [51]:
risk_df.to_csv(
    "../reports/risk_scoring.csv",
    index=False
)

print("\nSaved:")
print("../reports/risk_scoring.csv")

print("\nTotal sales at risk: ₹",
      round(risk_df["Sales_At_Risk_INR"].sum(), 2))

print("Total locked capital: ₹",
      round(risk_df["Locked_Capital_INR"].sum(), 2))

print("Total value at stake: ₹",
      round(risk_df["Value_At_Stake_INR"].sum(), 2))


Saved:
../reports/risk_scoring.csv

Total sales at risk: ₹ 52494085.13
Total locked capital: ₹ 9135994.23
Total value at stake: ₹ 61630079.37


In [52]:
print(risk_df.shape)
print(risk_df["SKU"].nunique())

print("\nForecast columns:")
print(future_forecast.columns.tolist())

print("\nReports created:")
from pathlib import Path
print([p.name for p in Path("../reports").glob("*")])

(50, 24)
50

Forecast columns:
['SKU', 'Date', 'Forecast_Units']

Reports created:
['forecast_model_evaluation.csv', 'future_6_week_forecast.csv', 'risk_scoring.csv']
